# CUDA scratch-memory experiment (FP64)

Start a fresh Kaggle NVIDIA GPU session with Internet enabled and at least 5 GiB free. This compares automatic shared/global scratch selection with forced-global scratch on identical CLQR fixtures. External solvers and JAX are disabled by default: no GTSAM, BLASFEO, or author-code builds. Regression tests and all four Compute Sanitizer checks remain enabled.
The dimension sweep uses N=128,512 and n=8,16,24,32,48,64, plus N=32,16384 at n=8,16. Ratios are m=n/2, p_s=n/4, p_m=n/8. This measures memory-placement cost, not a comparison with an older solver revision.

In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

# Editable defaults; existing environment values take precedence.
os.environ.setdefault("CLQR_RUN_EXTERNAL", "0")
os.environ.setdefault("CLQR_RUN_JAX", "0")
os.environ.setdefault("CLQR_RUN_ORIGINAL_TABLE", "0")
os.environ.setdefault("CLQR_RUN_TESTS", "1")
os.environ.setdefault("CLQR_RUN_SANITIZERS", "1")
revision = os.environ.get("CLQR_REVISION", "experiment/cuda-global-scratch")
work = Path("/kaggle/working")
source = Path(tempfile.mkdtemp(prefix="clqr-source-", dir=work))
url = "https://github.com/joaospinto/constrained_lqr_elimination.git"
subprocess.run(["git", "init", "-q", str(source)], check=True)
subprocess.run(["git", "-C", str(source), "remote", "add", "origin", url], check=True)
subprocess.run(["git", "-C", str(source), "fetch", "--depth=1", "--filter=blob:none",
                "origin", revision], check=True)
subprocess.run(["git", "-C", str(source), "-c", "advice.detachedHead=false",
                "checkout", "--detach", "FETCH_HEAD"], check=True)
print("Source revision:", subprocess.check_output(
    ["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip(), flush=True)
result = subprocess.run(["python3", "-u", str(source / "scripts/notebook_paper.py"),
                         "--work-dir", str(work), "--scratch-comparison", "--repeats", "11"])
print("Notebook driver exit code:", result.returncode)


Download the printed `paper-results.zip`. It contains `scratch_comparison.csv`, CPU/CUDA raw timings and errors for each mode, CPU/GPU identification and toolchain details, selected options, and test/sanitizer logs. Ratios above one mean forced-global is slower. Both builds share one private cache; the runner removes it after archiving results, including on an ordinary failure. Numerical benchmark failures remain rows in the CSV; test/sanitizer failures are reported separately.